In [ ]:
from __future__ import annotations

import operator
import os
from dotenv import load_dotenv
from pathlib import Path
from typing import TypedDict, List, Annotated, Literal

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal, Annotated
from typing_extensions import TypedDict
import operator
load_dotenv()
class Task(BaseModel):
    id: int
    title: str

    goal: str = Field(
        ...,
        description="One sentence describing what the reader should be able to do/understand after this section.",
    )

    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=5,
        description="3–5 concrete, non-overlapping subpoints to cover in this section.",
    )
    target_words: int = Field(
    ...,
    ge=120,
    le=450,
    description="Target word count for this section (120–450).",
)
    
    section_type: Literal[
    "intro",
    "problem",
    "core",
    "examples",
    "checklist",
    "common_mistakes",
    "conclusion",
    ] = Field(
        ...,
        description="Use 'common_mistakes' exactly once in the plan.",
    )


class Plan(BaseModel):
    blog_title: str
    audience: str = Field(..., description="Who this blog is for.")
    tone: str = Field(..., description="Writing tone (e.g., practical, crisp).")
    tasks: List[Task]


class State(TypedDict):
    topic: str
    plan: Plan
    sections: Annotated[List[str], operator.add]
    final: str

In [ ]:

llm = ChatGroq(model="llama-3.1-8b-instant",api_key=os.getenv("GROQ_API_KEY"))

In [34]:
def orchestrator(state: State) -> dict:
    planner = llm.with_structured_output(Plan)

    plan = planner.invoke(
        [
            SystemMessage(
                content=(
                    "You are a senior technical writer and developer advocate. "
                    "Your job is to produce a highly actionable outline for a technical blog post.\n\n"

                    "Hard requirements:\n"
                    "- Create 5–7 sections (tasks).\n"
                    "- Each section must include:\n"
                    "  1) goal (1 sentence)\n"
                    "  2) 3–5 concrete, non-overlapping bullets\n"
                    "  3) target word count between 120 and 450\n"
                    "- Include EXACTLY ONE section with section_type='common_mistakes'.\n\n"

                    "IMPORTANT:\n"
                    "- Use ONLY these section_type values:\n"
                    "  * intro\n"
                    "  * core\n"
                    "  * examples\n"
                    "  * checklist\n"
                    "  * common_mistakes\n"
                    "  * conclusion\n"
                    "- Never generate any other value for section_type.\n"
                    "- Every target_words value MUST be between 120 and 450.\n\n"

                    "Make it technical:\n"
                    "- Assume the reader is a developer.\n"
                    "- Build the blog in this order:\n"
                    "  intro → core → examples → common_mistakes → checklist → conclusion.\n"
                    "- Bullets must be actionable and testable.\n"
                    "- Include at least one minimal code example, edge case, debugging tip, "
                    "or performance consideration somewhere in the outline.\n"
                    "- Avoid vague bullets.\n\n"

                    "Output must strictly match the Plan schema."
                )
            ),
            HumanMessage(
                content=f"Topic: {state['topic']}"
            ),
        ]
    )

    return {"plan": plan}

In [35]:

def fanout(state: State):
    return [
        Send(
            "worker",
            {"task": task, "topic": state["topic"], "plan": state["plan"]},
        )
        for task in state["plan"].tasks
    ]

In [36]:
def worker(payload: dict) -> dict:

    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    bullets_text = "\n- " + "\n- ".join(task.bullets)

    section_md = llm.invoke(
        [
            SystemMessage(
    content=(
        "You are a senior technical writer and developer advocate. Write ONE section of a technical blog post in Markdown.\n\n"
        "Hard constraints:\n"
        "- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).\n"
        "- Stay close to the Target words (±15%).\n"
        "- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).\n\n"
        "Technical quality bar:\n"
        "- Be precise and implementation-oriented (developers should be able to apply it).\n"
        "- Prefer concrete details over abstractions: APIs, data structures, protocols, and exact terms.\n"
        "- When relevant, include at least one of:\n"
        "  * a small code snippet (minimal, correct, and idiomatic)\n"
        "  * a tiny example input/output\n"
        "  * a checklist of steps\n"
        "  * a diagram described in text (e.g., 'Flow: A -> B -> C')\n"
        "- Explain trade-offs briefly (performance, cost, complexity, reliability).\n"
        "- Call out edge cases / failure modes and what to do about them.\n"
        "- If you mention a best practice, add the 'why' in one sentence.\n\n"
        "Markdown style:\n"
        "- Start with a '## <Section Title>' heading.\n"
        "- Use short paragraphs, bullet lists where helpful, and code fences for code.\n"
        "- Avoid fluff. Avoid marketing language.\n"
        "- If you include code, keep it focused on the bullet being addressed.\n"
    )
)
,
            HumanMessage(
                content=(
                    f"Blog: {plan.blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Section type: {task.section_type}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Bullets:{bullets_text}\n"
                )
            ),
        ]
    ).content.strip()

    return {"sections": [section_md]}

In [37]:
def reducer(state: State) -> dict:

    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()
    
    final_md = f"# {title}\n\n{body}\n"

    # Save to file
    filename = "".join(c if c.isalnum() or c in (" ", "_", "-") else "" for c in title)
    filename = filename.strip().lower().replace(" ", "_") + ".md"
    Path(filename).write_text(final_md, encoding="utf-8")

    return {"final": final_md}

In [ ]:

# ----------------------------- 
# 5) Graph
# -----------------------------
g = StateGraph(State)
g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("reducer", reducer)

g.add_edge(START, "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()

app

out = app.invoke({"topic": "Write a blog on Self Attention", "sections": []})
print(out["final"])

# Mastering Self-Attention: A Deep Dive for Developers

## Introduction to Self-Attention

Self-attention is a fundamental concept in natural language processing (NLP) that enables models to focus on specific parts of the input when generating output. It plays a crucial role in transformer architecture, which has revolutionized the field of NLP.

### Basic Concept and Motivation

Self-attention allows a model to attend to different parts of the input simultaneously and weigh their importance. This is in contrast to traditional recurrent neural networks (RNNs), which process input sequentially. In NLP, self-attention is particularly useful for modeling long-range dependencies between words in a sentence.

### Key Components of Self-Attention

Self-attention consists of three main components:

* **Queries (Q)**: These are the input elements that we want to attend to.
* **Keys (K)**: These are the input elements that we want to attend from.
* **Values (V)**: These are the input elements t